# Criando epochs e gerando respostas evocados (ERP/ERF)

In [ ]:
import pathlib
import matplotlib

import mne
import mne_bids

matplotlib.use('Qt5Agg')

In [ ]:
bids_root = pathlib.Path('out_data/sample_BIDS')

bids_path = mne_bids.BIDSPath(subject='01',
                              session='01',
                              task='audiovisual',
                              run='01',
                              datatype='meg',
                              root=bids_root)

raw = mne_bids.read_raw_bids(bids_path)
raw.load_data() # Não podemos filtrar sem carregar os dados
raw.filter(l_freq=0.1, h_freq=40)
events, event_id = mne.events_from_annotations(raw)

In [ ]:
event_id

### Offset

>**OFFSET = altura errada do sinal (para cima ou para baixo) causada por fatores não neurais.**

É **ruído** e **não tem significado fisiológico**.


#### Como remover esse offset?

Usando **baseline**.

Baseline calcula a média do período pré-estímulo e subtrai essa média do epoch inteiro, removendo o offset.


#### Problema de escala?

O offset **NÃO é um problema de escala** - é um **deslocamento DC (corrente contínua)** causado por fatores físicos e técnicos:
---


In [ ]:
# Precisamos especificar onde um epoch começa e onde termina EM RELAÇÃO A EVENTOS NOS NOSSOS DADOS


tmin = -0.3 # Especificamos o tempo (em segundos) em relação ao evento e não ao dado inteiro
tmax = 0.5 # Richard falou que é importante que esteja em segundos e não em ms

'''
Então, para cada evento, você vai extrair uma janela que vai de -0.3 s até +0.5 s em relação ao momento exato do evento (time-lock)
'''

'''
É um trecho do epoch usado como referência zero.
Serve para remover offsets e variações lentas do sinal.
'''

# baseline = (início, fim), sempre relativo ao evento
# None significa "use o início do epoch"
# Portanto: baseline vai do começo do epoch (-0.3 s) até o evento (0 s)
baseline = (None, 0) # 'None' significa no começo da epoch, ou seja, a baseline é do começo da epoch até o começo do evento (event onset)



epochs = mne.Epochs(raw,
                    events=events,
                    event_id=event_id,
                    tmin=tmin,
                    tmax=tmax,
                    baseline=baseline,
                    preload=True)
'''
# Criamos os epochs a partir dos dados contínuos 'raw'
# events      -> matriz com: [amostra_do_evento, 0, código_evento]
# event_id    -> dicionário: nome_da_condição -> código_do_evento
# tmin/tmax   -> janela temporal do epoch
# baseline    -> período para correção
# preload=True faz o MNE carregar os dados dos epochs direto na memória
'''
epochs

### Baseline


É um trecho do epoch usado como **referência zero**.

Serve para remover:

* offsets do sinal
* drift lento
* diferenças iniciais entre trials
* variações que não têm relação com o evento


In [ ]:
epochs.plot()

# Fica com umas linhas tracejadas na janela interativa
# Quando marco uma parte ele marca na mesma posição em cada epoch
# Podemos marcar epochs como 'bad'
# Pressionando 'h' podemos ver um histograma das epochs

# Disclaimer: Quando você marca uma epoch como bad na visualização interativa (epochs.plot()), isso não é salvo automaticamente no objeto.
# Como posso salvar isso??

In [ ]:
epochs

<div class="alert alert-success">
    <b>EXERCÍCIO</b>:
    <ul>
        <li>Crie épocas começando 250 ms antes do início do estímulo e terminando 800 ms após o início do estímulo, aplicando correção de baseline com um período de baseline variando de -200 a 0 ms.</li>
    </ul>
</div>


In [ ]:
# ---Resposta --- #

tmin2 = -0.25
tmax2 = 0.8 # sempre relativo ao evento

baseline2 = (-0.20, 0) # Então vai 50ms depois da epoch até o início do evento. SEMPRE EM SEGUNDOS



epochs2 = mne.Epochs(raw,
                    events=events,
                    event_id=event_id,
                    tmin=tmin2,
                    tmax=tmax2,
                    baseline=baseline2,
                    preload=True)

epochs2


# Bem mais time points que a primeira versão

### Metadata?   

**Metadata** são **informações extras sobre cada época** - como "rótulos" ou "etiquetas" que descrevem características de cada trial.

#### Como adicionar Metadata?

**Método 1: Criar DataFrame e passar ao criar épocas**

```python
import pandas as pd
import numpy as np

# Criar DataFrame com informações sobre cada evento
# Cada linha = um trial/época
metadata = pd.DataFrame({
    'trial_number': range(1, len(events) + 1),
    'reaction_time': np.random.uniform(0.3, 0.8, len(events)),  # Exemplo
    'accuracy': np.random.choice(['correct', 'incorrect'], len(events)),
    'stimulus_type': np.random.choice(['A', 'B'], len(events))
})

print(metadata.head())
#    trial_number  reaction_time  accuracy stimulus_type
# 0             1       0.542        correct             A
# 1             2       0.678     incorrect             B
# 2             3       0.445        correct             A
# ...

# Criar épocas COM metadata
epochs = mne.Epochs(raw, 
                    events=events, 
                    event_id=event_id,
                    tmin=-0.3, 
                    tmax=0.5,
                    baseline=(None, 0),
                    metadata=metadata,  # ← Adiciona aqui!
                    preload=True)

# ✅ Agora não aparece mais "No metadata set"
```

#### **Método 2: Adicionar metadata depois**

```python
# Criar épocas normalmente
epochs = mne.Epochs(raw, events, event_id, 
                    tmin=-0.3, tmax=0.5, 
                    baseline=(None, 0))

# Criar metadata separadamente
metadata = pd.DataFrame({
    'condition': ['A', 'B', 'A', 'B'] * (len(epochs) // 4),
    'block': [1, 1, 2, 2] * (len(epochs) // 4)
})

# Adicionar aos epochs
epochs.metadata = metadata

```
---

## Selecionando epochs com base nas condições experimentais

Posso selecionar minhas epochs usando o nome do evento associado (é um dicionário!)

In [ ]:
epochs['Auditory/Right']

In [ ]:
epochs['Auditory'] # Posso Selecionar mais de um conjunto passando um termo geral (Auditory/Right e Auditory/Left)

In [ ]:
epochs['Left']

In [ ]:
epochs['Visual'].plot() # Posso só plotar minhas epochs específicas na visualização padrão


In [ ]:
epochs['Visual'].plot_image()

'''
Cria um gráfico para cada TIPO de sensor que existe nos seus dados e gera uma visualização tipo matriz (heatmap) das epochs.

- Serve para Ver padrões médios e consistência trial-by-trial.

- Eixo x → tempo relativo ao evento
- Eixo y → trials (epochs individuais)
- Cor → amplitude do sinal

É INTERATIVO! Dá pra ajustar a escala
'''

<div class="alert alert-success">
    <b>EXERCÍCIO</b>:
    <ul>
        <li>Extraia todas as épocas com a condição "Direita". Crie um gráfico e inclua apenas os canais de EEG.</li>
    </ul>
</div>

In [ ]:
# ---Resposta --- #

epochs_eeg = epochs.copy().pick('eeg') # in-place...
epochs_eeg['Right'].plot_image()

# Preciso criar uma cópia? SIM

## Salvando epochs

In [ ]:

epochs.save(pathlib.Path('out_data') / 'epochs_epo.fif',
            overwrite=True)
# Se eu usar uma abreviação diferente das padrões o mne reclama 'raw' ou 'epo' são exemplos da que devem ser usadas

'''
epochs.save(pathlib.Path('out_data') / 'epochs_xxx.fif',
            overwrite=True)
"This filename (out_data/epochs_xxx.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz"
'''


## Criando evoked data

- Evoked data = média de várias epochs
- Evoked = "Evocado" = "Provocado": a resposta cerebral evocada/provocada por um estímulo externo.
- **ERP e ERF são tipos de evoked data**

> **Evoked revela a resposta cerebral consistente a um estímulo, cancelando o ruído aleatório.**



| Aspecto | Epochs | Evoked |
|---------|--------|--------|
| **O que é** | Tentativas individuais | Média das tentativas |
| **Shape** | (n_epochs, n_channels, n_times) | (n_channels, n_times) |
| **Qualidade** | Ruidoso | Limpo |
| **Uso** | Análise single-trial, ML | Análise de grupo, ERPs clássicos |
| **Criar** | `mne.Epochs()` | `epochs.average()` |


In [ ]:
evoked_auditory = epochs['Auditory'].average()
evoked_visual = epochs['Visual'].average()

evoked_auditory # por default pega 50% de cada (right e left)

In [ ]:
evoked_auditory.plot_topomap(ch_type='mag') # Plota mapas topográficos ao longo do tempo mostrando onde a resposta auditiva aparece no couro cabeludo 

'''
Mapas topográficos (topoplots - É um “mapa de calor” da intensidade do sinal espalhada pela cabeça) da atividade neural 
ao longo do couro cabeludo


MEG possui dois tipos de sensores:
mag → magnetômetros (medem campo magnético diretamente)
grad → gradiômetros (medem gradiente do campo magnético)

'''



In [ ]:
# Posso customizar meus intervalos

evoked_auditory.plot_topomap(ch_type='mag', times = [0, 0.05, 0.100, 0.8]) # se eu colocar um que não existe ele quebra obviamente

In [ ]:
evoked_auditory.plot_joint(picks='mag') # MUITO útil

'''
combina 3 tipos de gráficos ao mesmo tempo:

- A forma de onda (Evoked) ao longo do tempo
- Topomaps em momentos-chave do sinal
- Picos automaticamente detectados (partes em destaque)
'''

In [ ]:
# Posso comparar duas (ou mais) respostas Evoked no mesmo gráfico

# Como em outros plots posso manipular esse gráfico, mudando as cores, desenhos das linhas, etc.
mne.viz.plot_compare_evokeds([evoked_auditory, evoked_visual], picks='mag')


'''
Ele coloca as duas curvas Evoked no mesmo eixo, alinhadas no tempo, para que você possa comparar:

- amplitude
- latência dos picos
- forma da onda
- diferenças de polaridade
- topografia (opcional, se ativado)
'''

<div class="alert alert-success">
    <b>EXERCÍCIO</b>:
    <ul>
        <li>Plote uma comparação de GFP para as condições "Visual/Left" e "Visual/Right" dos dados de EEG.</li>
    </ul>
</div>


In [ ]:
# ---Resposta --- #

evoked_visual2 = (epochs['Visual/Right'].average(),
                  epochs['Visual/Left'].average())

mne.viz.plot_compare_evokeds([evoked_visual2[0], evoked_visual2[1]], picks='eeg')

## Salvando evoked data

In [ ]:
# Função do MNE usada para salvar Evoked objects em um arquivo FIF

mne.write_evokeds(fname=pathlib.Path('out_data') / 'evokeds_ave.fif',
                  evoked=[evoked_auditory, evoked_visual], overwrite=True)

| Parte                        | Função                             |
| ---------------------------- | ---------------------------------- |
| `mne.write_evokeds`          | salva Evokeds em disco             |
| `fname`                      | caminho do arquivo `.fif`          |
| `evoked=[...]`               | lista de Evoked a serem salvos     |
| `pathlib.Path(...) / 'file'` | cria caminho seguro para o arquivo |


## Lendo evoked data

In [ ]:
evokeds = mne.read_evokeds(fname=pathlib.Path('out_data') / 'evokeds_ave.fif')
evokeds

In [ ]:
evokeds[0] 

In [ ]:
# Usamos o parâmetro 'condition' para especificar, pois read_evokeds() sem condition retorna todos.

evoked = mne.read_evokeds(fname=pathlib.Path('out_data') / 'evokeds_ave.fif',
                          condition='0.50 × Auditory/Left + 0.50 × Auditory/Right')
evoked